# Monthly control panel

Thin panel over `core.monthly` (the same functions GitHub Actions runs as `python -m core.monthly ...`). No logic here.
Order: 1 config, 2 plans (nothing is sent), 3 scrape, 4 TÜV SÜD, 5 build by hand (timeout case), 6 status, 7 folders.

`SIMULATED = True` runs every source in its simulated mode in a throwaway sandbox folder and writes the master to its `_dev/`: no request, nothing in the real data. `SIMULATED = False` works on the real data root: every step then needs `CONFIRM = True` on purpose and sends live requests (GTÜ ~900, KÜS ~182, ...). Personal data is in the run folders: keep them out of git and chat.

## 1. Config

In [ ]:
from core import monthly, paths
from core.geo import ensure_geonames, load_plz_centroids
import tempfile

SIMULATED = True  # True: simulated sources in a sandbox; False: the real data root (live requests!)
CONFIRM = SIMULATED  # False: every command only prints its plan; set True by hand for a live step

centroids = load_plz_centroids(ensure_geonames(paths.geonames_dir()))
ROOT = tempfile.mkdtemp(prefix='monthly_sandbox_') if SIMULATED else None
ctx = monthly.Context(root=ROOT, simulated=SIMULATED, centroids=centroids, isolation='inline' if SIMULATED else 'process',
                      master_kwargs={'centroids': centroids} if SIMULATED else {})
print('root:', ctx.root, '| simulated:', ctx.simulated, '| confirm:', CONFIRM)

## 2. Plans (nothing is sent, nothing is written)

In [ ]:
monthly.scrape(ctx)
monthly.tuev_sued(ctx, 'plan')
monthly.build(ctx);

## 3. Scrape (the 1st)
HDP and every source except TÜV SÜD; passed sources are skipped; builds the master at once if TÜV SÜD is fresh.

In [ ]:
FORCE = []  # e.g. ['dekra'] reruns one source that already passed this month
res = monthly.scrape(ctx, confirm=CONFIRM, force=FORCE)
print('exit', res.code, '|', res.status['phase'] if res.status else res.message)

## 4. TÜV SÜD
Simulated: the whole offline flow. Live: `plan` writes the snippet; set `DOWNLOAD` to the uploaded .jsonl to check it (PASS builds the run and, if the scrape is done, the master; gaps write the next snippet).

In [ ]:
DOWNLOAD = None  # live only: path of the downloaded .jsonl
if SIMULATED:
    res = monthly.tuev_sued_simulated(ctx)
elif DOWNLOAD:
    res = monthly.tuev_sued(ctx, 'check', confirm=CONFIRM, file=DOWNLOAD)
else:
    res = monthly.tuev_sued(ctx, 'plan', confirm=CONFIRM)
print('exit', res.code, '|', res.status['phase'] if res.status else res.message)

## 5. Build by hand
Only for the timeout (`WITHOUT_TUEV_SUED = True`) or after a fix. `ACCEPT_RETIREMENTS` is a human decision (a source really lost more than 10 % of its sites).

In [ ]:
WITHOUT_TUEV_SUED = False
ACCEPT_RETIREMENTS = False
res = monthly.build(ctx, confirm=CONFIRM, without_tuev_sued=WITHOUT_TUEV_SUED, accept_retirements=ACCEPT_RETIREMENTS)
print('exit', res.code, '|', res.message or (res.status or {}).get('phase'))

## 6. Status
Rebuilt from the run folders (no requests); `confirm=True` also rewrites the status file.

In [ ]:
res = monthly.status(ctx)

## 7. Folders

In [ ]:
import os

OPEN = False  # True opens them in Explorer
for f in monthly.folders(ctx):
    print(f)
    if OPEN:
        os.startfile(f)